# NLP 4-bit sentiment classifier (alpha_ctrl NLP mapping)

Tiny 4-bit text classifier on **SST-2** (binary sentiment). One encoder block; its second feed-forward
layer `ff2` (64 -> 64, a 1x1 `QuantConv2d`, 4-bit act / 4-bit weight, no bias, followed directly by a ReLU)
is the layer that runs on the 8x8 systolic array. Tokens are the array's "pixels": one sentence of
T tokens = one IC=64, OC=64, K=1, H=1, W=T layer for `core_tiled_tb.v`.

Flow (same idea as `VGGNet_Quant_Training.ipynb`):
1. Cell 0: setup, data, helpers. 2. Train (GPU). 3. Load + test accuracy.
4-6. Extract `x_int`, `w_int` for one sentence and check the integer layer against PyTorch.
7. Save the final model. 8-9. Export sentences in the hardware file format.
10. Feed hardware results back into the model. 11-13. 2:4 pruning + fine-tune + export for `row = 16`.

Training needs a GPU server and `pip install datasets` (downloads GLUE SST-2 once). Export / feed-back
run fine on CPU: copy the trained checkpoint back and set `USE_GPU = False`.

In [ ]:
import os
import re
import time
import shutil
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from models.quant_layer import act_quantization
from models.nlp_quant import *
from nlp_export import write_sentence, write_weight, read_psum_file, check_layer, check_nm

global best_prec
use_gpu = torch.cuda.is_available()
device = torch.device("cuda" if use_gpu else "cpu")
print('Using device:', device)

torch.manual_seed(0)
np.random.seed(0)

batch_size = 128
MAX_LEN = 64           # tokens per sentence (= pixels per pass; L0 / OFIFO depth is 64)
MIN_FREQ = 2           # words seen fewer times in training map to <unk>
DROPOUT = 0.3          # regularization: the model overfits SST-2 at 0.1 (train 96% vs test ~81%)
LABEL_SMOOTHING = 0.1  # training loss only; keeps the model from becoming over-confident
model_name = "SentimentNet_quant_64"


######### Data: GLUE SST-2 #########
# test split labels are hidden, so the 872-sentence validation split is used as the test set
from datasets import load_dataset
sst2 = load_dataset('glue', 'sst2')

def tokenize(s):
    return re.findall(r"[a-z0-9']+|[^\sa-z0-9']", s.lower())

counts = Counter(w for s in sst2['train']['sentence'] for w in tokenize(s))
vocab = ['<pad>', '<unk>'] + sorted(w for w, c in counts.items() if c >= MIN_FREQ)
stoi = {w: i for i, w in enumerate(vocab)}
print('vocab size:', len(vocab))

def encode(s):
    ids = [stoi.get(w, 1) for w in tokenize(s)][:MAX_LEN]
    return ids if ids else [1]

class SST2(torch.utils.data.Dataset):
    def __init__(self, split):
        self.x = [encode(s) for s in sst2[split]['sentence']]
        self.y = list(sst2[split]['label'])
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.x[i], self.y[i]

def collate(batch):
    T = max(len(x) for x, _ in batch)
    tokens = torch.zeros(len(batch), T, dtype=torch.long)       # 0 = <pad>
    for i, (x, _) in enumerate(batch):
        tokens[i, :len(x)] = torch.tensor(x)
    return tokens, torch.tensor([y for _, y in batch])

train_dataset = SST2('train')
test_dataset = SST2('validation')
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, collate_fn=collate)
testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, collate_fn=collate)
print('train / test sentences:', len(train_dataset), len(test_dataset),
      '| max tokens:', max(len(x) for x in train_dataset.x))

print('=> Building model...')
model = SentimentNet_quant_64(len(vocab), max_len=MAX_LEN, dropout=DROPOUT)


print_freq = 100 # every 100 batches, accuracy printed. Here, each batch includes "batch_size" data points

def train(trainloader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter()
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()
    for i, (input, target) in enumerate(trainloader):
        data_time.update(time.time() - end)
        input, target = input.to(device), target.to(device)

        output = model(input)
        loss = criterion(output, target)

        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_time.update(time.time() - end)
        end = time.time()

        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time, loss=losses, top1=top1))


def validate(val_loader, model, criterion):
    losses = AverageMeter()
    top1 = AverageMeter()
    model.eval()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
            input, target = input.to(device), target.to(device)
            output = model(input)
            loss = criterion(output, target)
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))
    print(' * Prec {top1.avg:.3f}%  Loss {loss.avg:.4f}'.format(top1=top1, loss=losses))
    return top1.avg


def accuracy(output, target, topk=(1,)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk)
    batch_size = target.size(0)
    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))
    res = []
    for k in topk:
        correct_k = correct[:k].reshape(-1).float().sum(0)
        res.append(correct_k.mul_(100.0 / batch_size))
    return res


class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()
    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


def save_checkpoint(state, is_best, fdir):
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)
    if is_best:
        shutil.copyfile(filepath, os.path.join(fdir, 'model_best.pth.tar'))


def calibrate_act_alpha(model, pct=99.9, n_batches=50):
    """Set ff2.act_alpha to the pct-th percentile of ff2's inputs on training data.

    quant_layer's act_alpha only gets a gradient from clipped activations, so if it starts too
    large (default 8.0; ff2's inputs are ~1 after LayerNorm + ff1) it never shrinks and most of
    the 16 levels stay unused. Calibrating once after the first epoch fixes the range; training
    then continues normally."""
    model.eval()
    vals = []
    with torch.no_grad():
        for i, (input, _) in enumerate(trainloader):
            if i == n_batches:
                break
            input = input.to(device)
            _, ff2_in, mask = model.pre(input)
            vals.append(ff2_in[:, :, 0, :].permute(0, 2, 1)[mask].flatten().cpu())
    alpha = torch.quantile(torch.cat(vals)[:1_000_000], pct / 100).item()
    model.ff2.act_alpha.data.fill_(alpha)
    print(f'calibrated ff2.act_alpha = {alpha:.4f} ({pct}th percentile of ff2 inputs)')
    model.train()


def train_loop(model, epochs, lr, weight_decay, fdir, calibrate_after=None):
    """AdamW + cosine schedule; keeps the best test accuracy in fdir/model_best.pth.tar.
    calibrate_after=k: calibrate ff2.act_alpha after epoch k (1 = after the first epoch)."""
    best_prec = 0
    model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    os.makedirs(fdir, exist_ok=True)
    for epoch in range(0, epochs):
        train(trainloader, model, criterion, optimizer, epoch)
        if calibrate_after is not None and epoch + 1 == calibrate_after:
            calibrate_act_alpha(model)
        print("Validation starts")
        prec = validate(testloader, model, criterion)
        scheduler.step()
        is_best = prec > best_prec
        best_prec = max(prec, best_prec)
        print('best acc: {:1f}'.format(best_prec))
        save_checkpoint({
            'epoch': epoch + 1,
            'state_dict': model.state_dict(),
            'best_prec': best_prec,
            'optimizer': optimizer.state_dict(),
            'vocab': vocab,
        }, is_best, fdir)
    return best_prec

In [ ]:
print(model)
print('parameters:', sum(p.numel() for p in model.parameters()))
print('hardware layer:', model.ff2)

In [ ]:
# TRAIN SENTIMENT QUANT MODEL (dense)

# Hyperparameters
lr = 2e-3
weight_decay = 0.05    # AdamW decoupled decay; mostly regularizes the 858k embedding weights
epochs = 10

# calibrate_after=1: fix ff2's activation range after the first epoch (see calibrate_act_alpha)
best_prec = train_loop(model, epochs, lr, weight_decay, 'result_nlp/', calibrate_after=1)

In [ ]:
USE_GPU = False  # True = GPU, False = CPU

FINAL_PATH = "final_trained_model/SentimentNet_quant_64_model.pth"
TRAIN_PATH = "result_nlp/model_best.pth.tar"

PATH = FINAL_PATH if os.path.exists(FINAL_PATH) else TRAIN_PATH
print("Loading:", PATH)

device = torch.device("cuda" if USE_GPU and torch.cuda.is_available() else "cpu")
print("Using device:", device)

checkpoint = torch.load(PATH, map_location=device, weights_only=False)
assert checkpoint['vocab'] == vocab, 'checkpoint vocab differs from the one built in cell 0'
model.load_state_dict(checkpoint['state_dict'])
model = model.to(device)
model.eval()

correct = 0
with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device)
        pred = model(data).argmax(dim=1)
        correct += pred.eq(target).sum().item()

print('\nTest set: Accuracy: {}/{} ({:.2f}%)\n'.format(
    correct, len(testloader.dataset), 100. * correct / len(testloader.dataset)))
model.show_params()

In [ ]:
class SaveOutput:
    def __init__(self):
        self.outputs = []
    def __call__(self, module, module_in):
        self.outputs.append(module_in)
    def clear(self):
        self.outputs = []

######### Save the input of ff2 and of the ReLU after it, for one sentence ##########
SENT_IDX = 0
tokens = torch.tensor([test_dataset.x[SENT_IDX]], device=device)   # batch of 1, no padding: T = sentence length
T = tokens.size(1)
print('sentence:', sst2['validation'][SENT_IDX]['sentence'])
print('label:', test_dataset.y[SENT_IDX], '| tokens:', T)

save_ff2_in = SaveOutput()
save_ff2_out = SaveOutput()
h1 = model.ff2.register_forward_pre_hook(save_ff2_in)
h2 = model.relu2.register_forward_pre_hook(save_ff2_out)
with torch.no_grad():
    out = model(tokens)
h1.remove()
h2.remove()

x = save_ff2_in.outputs[0][0][0]           # [64, 1, T], float, after ReLU (>= 0)
next_in = save_ff2_out.outputs[0][0][0]    # [64, 1, T], ff2 output before its ReLU

#### Convert x into x_int ####
x_bit = 4
x_alpha = model.ff2.act_alpha
x_delta = x_alpha / (2 ** x_bit - 1)

act_quant_fn = act_quantization(x_bit)
x_q = act_quant_fn(x, x_alpha)
x_int = (x_q / x_delta).round()
print("x_int shape:", x_int.shape, "range:", int(x_int.min()), "..", int(x_int.max()), "(should reach ~15)")

In [ ]:
weight_q = model.ff2.weight_q
w_alpha = model.ff2.weight_quant.wgt_alpha
w_bit = 4

w_delta = w_alpha / (2 ** (w_bit - 1) - 1)
w_int = (weight_q / w_delta).round()       # round to fix floating point error
print("w_int shape:", w_int.shape, "range:", int(w_int.min()), "..", int(w_int.max()))

In [ ]:
# Integer layer vs PyTorch: psum_int = w_int @ x_int, then rescale
conv_int = torch.nn.Conv2d(in_channels=64, out_channels=64, kernel_size=1, bias=False).to(device)
conv_int.weight = torch.nn.Parameter(w_int)

with torch.no_grad():
    psum_int = conv_int(x_int.unsqueeze(0))[0]            # integer layer, [64, 1, T]
    psum_recovered = psum_int * x_delta * w_delta          # rescale to real units

difference = abs(psum_recovered - next_in)
print('mean |difference| (pre-ReLU):', difference.mean().item())
print('max |psum_int|:', int(psum_int.abs().max()), '(must be < 32768)')

check_layer(x_int[:, 0].detach().cpu().numpy(), w_int[:, :, 0, 0].detach().cpu().numpy(),
            psum_int[:, 0].detach().cpu().numpy())
print('ranges OK for the hardware')

In [ ]:
# Save the final model without the optimizer (like the VGG flow)
best = torch.load('result_nlp/model_best.pth.tar', map_location='cpu', weights_only=False)
os.makedirs('final_trained_model', exist_ok=True)
torch.save({'state_dict': best['state_dict'],
            'best_prec':  best['best_prec'],
            'epoch':      best['epoch'],
            'vocab':      best['vocab']},
           'final_trained_model/SentimentNet_quant_64_model.pth')
print('saved, best acc {:.2f}%'.format(best['best_prec']))

## Export for the hardware

Layout (see `nlp_export.py`): `nlp_hw/<variant>/weight.txt` once, and one folder per sentence
`nlp_hw/<variant>/sent_XXXX/` with `input.txt`, `psum.txt` (expected ReLU output) and `plusargs.txt`
(`+IC=64 +OC=64 +K=1 +H=1 +W=<tokens>`). `meta.npz` holds the scales, labels and PyTorch predictions
for the feed-back step.

To run one sentence on the hardware now (dense build), from a copy of `hardware/alpha_ctrl`:
```
cp nlp_hw/dense_row8/weight.txt nlp_hw/dense_row8/sent_0000/{input,psum}.txt <copy>/sys_data/
iverilog -g2012 -DUSE_CTRL -s core_tiled_tb -o sim -c filelist_tiled
vvp sim $(cat nlp_hw/dense_row8/sent_0000/plusargs.txt)        # expect "No error detected"
```

In [ ]:
def layer_ints(model, tokens):
    """One sentence (tokens [1, T], no padding) -> (h, mask, x_int [64, T], w_int [64, 64], deltas)."""
    with torch.no_grad():
        h, ff2_in, mask = model.pre(tokens)
        x_delta = (model.ff2.act_alpha / 15).item()
        w_delta = (model.ff2.weight_quant.wgt_alpha / 7).item()
        x_q = act_quantization(4)(ff2_in, model.ff2.act_alpha)
        x_int = (x_q / x_delta).round()[0, :, 0, :]                          # [64, T]
        weight_q = model.ff2.weight_quant(model.ff2.weight) * model.ff2.nm_mask
        w_int = (weight_q / w_delta).round()[:, :, 0, 0]                     # [64, 64]
    return h, mask, x_int, w_int, x_delta, w_delta


def export_batch(model, out_dir, n_sent, row=8, nm=False):
    """Write weight.txt + n_sent sentence folders + meta.npz."""
    model.eval()
    os.makedirs(out_dir, exist_ok=True)
    lens, labels, preds = [], [], []
    for s in range(n_sent):
        tokens = torch.tensor([test_dataset.x[s]], device=device)
        h, mask, x_int, w_int, x_delta, w_delta = layer_ints(model, tokens)
        psum_int = w_int @ x_int                                             # [64, T] integer
        check_layer(x_int.cpu().numpy(), w_int.cpu().numpy(), psum_int.cpu().numpy())
        if s == 0:
            if nm:
                assert check_nm(w_int.cpu().numpy().astype(int)), 'ff2 weights are not 2:4'
            write_weight(f'{out_dir}/weight.txt', w_int, row=row, nm=nm)
        write_sentence(f'{out_dir}/sent_{s:04d}', x_int, psum_int, row=row)
        with torch.no_grad():
            preds.append(model(tokens).argmax(1).item())
        lens.append(tokens.size(1))
        labels.append(test_dataset.y[s])
    np.savez(f'{out_dir}/meta.npz', lens=np.array(lens), labels=np.array(labels),
             torch_preds=np.array(preds), x_delta=x_delta, w_delta=w_delta, row=row, nm=nm)
    acc = 100. * np.mean(np.array(preds) == np.array(labels))
    print(f'{out_dir}: {n_sent} sentences, {sum(lens)} tokens, PyTorch accuracy on them {acc:.2f}%')


N_SENT = 872     # all of the SST-2 validation split; use e.g. 200 for a quicker hardware run
export_batch(model, 'nlp_hw/dense_row8', N_SENT, row=8, nm=False)

In [ ]:
def feedback(model, out_dir, result_name='hw_out.txt'):
    """Splice the hardware ff2 result back into the model and compare with pure PyTorch.

    For each sentence it reads sent_XXXX/<result_name> (hardware output, psum.txt format); if that
    file is missing it falls back to psum.txt (the expected result), which checks this code path
    before any hardware run."""
    meta = np.load(f'{out_dir}/meta.npz')
    x_delta, w_delta = float(meta['x_delta']), float(meta['w_delta'])
    model.eval()
    hw_preds, n_hw = [], 0
    for s, T in enumerate(meta['lens']):
        d = f'{out_dir}/sent_{s:04d}'
        f = f'{d}/{result_name}'
        if os.path.exists(f):
            n_hw += 1
        else:
            f = f'{d}/psum.txt'
        relu_int = read_psum_file(f, int(T))                                  # [64, T], already ReLU'd
        tokens = torch.tensor([test_dataset.x[s]], device=device)
        with torch.no_grad():
            h, _, mask = model.pre(tokens)
            y = torch.tensor(relu_int, dtype=torch.float32, device=device) * x_delta * w_delta
            logits = model.post(h, y.view(1, 64, 1, int(T)), mask)
        hw_preds.append(logits.argmax(1).item())
    hw_preds = np.array(hw_preds)
    same = int((hw_preds == meta['torch_preds']).sum())
    print(f'{out_dir}: {n_hw}/{len(hw_preds)} sentences used hardware results ({result_name})')
    print(f'  predictions identical to PyTorch: {same}/{len(hw_preds)}')
    print(f'  accuracy: hardware-in-the-loop {100*np.mean(hw_preds == meta["labels"]):.2f}%  '
          f'PyTorch {100*np.mean(meta["torch_preds"] == meta["labels"]):.2f}%')

feedback(model, 'nlp_hw/dense_row8')

## 2:4 sparse ff2

Prune `ff2` 2:4 along input channels (in every group of 4 consecutive input channels, per output
channel, keep the 2 largest weights), fine-tune with the mask fixed, and export compressed weights
for the `row = 16`, `SPARSE_NM = 1` build (8x8 compressed array, 16 input channels per tile).

In [ ]:
# PRUNE + FINE-TUNE (starts from the trained dense model loaded above)
model.ff2.set_nm_mask()
model.to(device)
with torch.no_grad():
    w = model.ff2.weight_quant(model.ff2.weight) * model.ff2.nm_mask
print('ff2 zero weights after pruning: {:.1f}%'.format(100 * (w == 0).float().mean().item()))
print('accuracy right after pruning (before fine-tune):')
validate(testloader, model, nn.CrossEntropyLoss().to(device))

nm_best = train_loop(model, epochs=3, lr=2e-4, weight_decay=0.05, fdir='result_nlp_nm/')

In [ ]:
# Load the best 2:4 model, check accuracy and the 2:4 pattern
ckpt = torch.load('result_nlp_nm/model_best.pth.tar', map_location=device, weights_only=False)
model.load_state_dict(ckpt['state_dict'])
model = model.to(device)
model.eval()

correct = 0
with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device)
        correct += model(data).argmax(dim=1).eq(target).sum().item()
print('2:4 test accuracy: {:.2f}%'.format(100. * correct / len(test_dataset)))

_, _, _, w_int_nm, _, _ = layer_ints(model, torch.tensor([test_dataset.x[0]], device=device))
print('2:4 pattern holds on the integer weights:', check_nm(w_int_nm.cpu().numpy().astype(int)))

torch.save({'state_dict': ckpt['state_dict'], 'best_prec': ckpt['best_prec'],
            'epoch': ckpt['epoch'], 'vocab': ckpt['vocab']},
           'final_trained_model/SentimentNet_quant_64_nm24_model.pth')

In [ ]:
# Export for the row=16 SPARSE_NM=1 build, then feed-back (psum.txt stand-in until hardware runs)
export_batch(model, 'nlp_hw/nm24_row16', N_SENT, row=16, nm=True)
feedback(model, 'nlp_hw/nm24_row16')